In [ ]:
# ==========================================================
# Section 1: Project Configuration & Dataset Loading
# ==========================================================
#
# Purpose
# -------
# Initialise the project environment and load the frozen
# multi-task datasets prepared in the previous notebooks.
#
# This section:
#   • Configures project directories
#   • Sets reproducibility parameters
#   • Detects the training device
#   • Configures logging
#   • Loads frozen train/validation/test datasets
#   • Loads class weights and metadata
#   • Performs initial validation
#
# Inputs
# ------
# data/processed/multitask/
#
#     multitask_train.csv
#     multitask_validation.csv
#     multitask_test.csv
#     multitask_class_weights.json
#     multitask_metadata.json
#
# Outputs
# -------
# In-memory datasets ready for tokenisation.
#
# ==========================================================

from pathlib import Path
import json
import logging
import random

import numpy as np
import pandas as pd
import torch

# ==========================================================
# Project Structure
# ==========================================================

PROJECT_ROOT = Path.cwd().resolve()

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = DATA_DIR / "processed"

MULTITASK_DIR = PROCESSED_DIR / "multitask"

MODELS_DIR = PROJECT_ROOT / "models"

BASELINE_DIR = MODELS_DIR / "baseline"

REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [

    MODELS_DIR,

    BASELINE_DIR,

    REPORTS_DIR,

]:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

# ==========================================================
# Reproducibility
# ==========================================================

RANDOM_STATE = 42

random.seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(RANDOM_STATE)

# Improve reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==========================================================
# Device Configuration
# ==========================================================

if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

elif torch.backends.mps.is_available():

    DEVICE = torch.device("mps")

else:

    DEVICE = torch.device("cpu")

# ==========================================================
# Training Configuration
# ==========================================================

MODEL_NAME = "xlm-roberta-base"

MAX_LENGTH = 128

BATCH_SIZE = 16

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

NUM_EPOCHS = 5

EARLY_STOPPING_PATIENCE = 2

GRADIENT_CLIP = 1.0

BASELINE_ALPHA = 1.0      # misinformation loss

BASELINE_BETA = 1.0       # hate speech loss

# ==========================================================
# Logging
# ==========================================================

logging.basicConfig(

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s"

)

logger = logging.getLogger(__name__)

logger.info("=" * 70)
logger.info("BASELINE MULTI-TASK TRAINING")
logger.info("=" * 70)

logger.info(f"Device           : {DEVICE}")
logger.info(f"Model            : {MODEL_NAME}")
logger.info(f"Batch size       : {BATCH_SIZE}")
logger.info(f"Learning rate    : {LEARNING_RATE}")
logger.info(f"Epochs           : {NUM_EPOCHS}")
logger.info(f"Random state     : {RANDOM_STATE}")

# ==========================================================
# Verify Required Files
# ==========================================================

required_files = {

    "Train Dataset":

        MULTITASK_DIR /

        "multitask_train.csv",

    "Validation Dataset":

        MULTITASK_DIR /

        "multitask_validation.csv",

    "Test Dataset":

        MULTITASK_DIR /

        "multitask_test.csv",

    "Class Weights":

        MULTITASK_DIR /

        "multitask_class_weights.json",

    "Metadata":

        MULTITASK_DIR /

        "multitask_metadata.json"

}

missing_files = [

    name

    for name, path in required_files.items()

    if not path.exists()

]

if missing_files:

    raise FileNotFoundError(

        "The following required files were not found:\n\n"

        + "\n".join(missing_files)

        + "\n\nRun 03_build_multitask_dataset.ipynb first."

    )

logger.info("✓ All required files located.")

# ==========================================================
# Load Frozen Datasets
# ==========================================================

logger.info("=" * 70)
logger.info("LOADING MULTI-TASK DATASETS")
logger.info("=" * 70)

train_df = pd.read_csv(

    required_files["Train Dataset"]

)

validation_df = pd.read_csv(

    required_files["Validation Dataset"]

)

test_df = pd.read_csv(

    required_files["Test Dataset"]

)

logger.info(f"Training samples    : {len(train_df):,}")

logger.info(f"Validation samples  : {len(validation_df):,}")

logger.info(f"Test samples        : {len(test_df):,}")

# ==========================================================
# Load Metadata
# ==========================================================

with open(

    required_files["Metadata"],

    "r",

    encoding="utf-8"

) as fp:

    dataset_metadata = json.load(fp)

logger.info("✓ Dataset metadata loaded.")

# ==========================================================
# Load Class Weights
# ==========================================================

with open(

    required_files["Class Weights"],

    "r",

    encoding="utf-8"

) as fp:

    multitask_class_weights = json.load(fp)

logger.info("✓ Class weights loaded.")

# ==========================================================
# Initial Dataset Validation
# ==========================================================

logger.info("=" * 70)
logger.info("INITIAL DATA VALIDATION")
logger.info("=" * 70)

required_columns = [

    "text",

    "dataset",

    "task",

    "task_label"

]

for split_name, dataframe in {

    "Train": train_df,

    "Validation": validation_df,

    "Test": test_df

}.items():

    missing_columns = [

        column

        for column in required_columns

        if column not in dataframe.columns

    ]

    if missing_columns:

        raise ValueError(

            f"{split_name} dataset is missing columns:\n"

            f"{missing_columns}"

        )

logger.info("✓ Required columns verified.")

# ==========================================================
# Dataset Overview
# ==========================================================

overview = pd.DataFrame({

    "Split": [

        "Train",

        "Validation",

        "Test"

    ],

    "Samples": [

        len(train_df),

        len(validation_df),

        len(test_df)

    ]

})

print("\nDataset Summary\n")

print(overview)

print("\nTraining Dataset Distribution\n")

print(

    train_df["dataset"]

    .value_counts()

)

logger.info("=" * 70)
logger.info("SECTION 1 COMPLETE")
logger.info("=" * 70)

logger.info(

    "Frozen datasets successfully loaded."

)

logger.info(

    "Ready for tokenisation."

)

In [ ]:
# ==========================================================
# Section 2: Tokenisation & Dataset Construction
# ==========================================================
#
# Purpose
# -------
# Tokenise the frozen multi-task datasets and construct
# PyTorch Dataset and DataLoader objects for model training.
#
# This section:
#   • Loads the XLM-RoBERTa tokenizer
#   • Defines the Multi-task Dataset class
#   • Tokenises text
#   • Creates PyTorch DataLoaders
#
# Outputs
# -------
# train_loader
# validation_loader
# test_loader
#
# ==========================================================

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

logger.info("=" * 70)
logger.info("TOKENISATION & DATASET CONSTRUCTION")
logger.info("=" * 70)

# ==========================================================
# Load Tokenizer
# ==========================================================

logger.info(f"Loading tokenizer: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

logger.info("✓ Tokenizer loaded successfully.")

# ==========================================================
# Multi-task Dataset
# ==========================================================

class MultiTaskDataset(Dataset):
    """
    PyTorch Dataset for the multi-task learning framework.

    Each sample contains:

        input_ids
        attention_mask
        task_id
        task_label
        dataset

    Task IDs

        0 -> Misinformation Detection
        1 -> Hate Speech Detection
    """

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length
    ):

        self.dataframe = dataframe.reset_index(drop=True)

        self.tokenizer = tokenizer

        self.max_length = max_length

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        encoding = self.tokenizer(

            row["text"],

            max_length=self.max_length,

            padding="max_length",

            truncation=True,

            return_attention_mask=True,

            return_tensors="pt"

        )

        return {

            "input_ids":

                encoding["input_ids"].squeeze(0),

            "attention_mask":

                encoding["attention_mask"].squeeze(0),

            "task_id":

                torch.tensor(

                    row["task_id"],

                    dtype=torch.long

                ),

            "task_label":

                torch.tensor(

                    row["task_label"],

                    dtype=torch.long

                ),

            "dataset":

                row["dataset"]

        }

# ==========================================================
# Construct Datasets
# ==========================================================

logger.info("Constructing PyTorch datasets...")

train_dataset = MultiTaskDataset(

    train_df,

    tokenizer,

    MAX_LENGTH

)

validation_dataset = MultiTaskDataset(

    validation_df,

    tokenizer,

    MAX_LENGTH

)

test_dataset = MultiTaskDataset(

    test_df,

    tokenizer,

    MAX_LENGTH

)

logger.info("✓ Dataset construction complete.")

# ==========================================================
# Construct DataLoaders
# ==========================================================

logger.info("Creating DataLoaders...")

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

validation_loader = DataLoader(

    validation_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=torch.cuda.is_available()

)

logger.info("✓ DataLoaders created.")

# ==========================================================
# Sanity Check
# ==========================================================

logger.info("=" * 70)
logger.info("TOKENISATION SANITY CHECK")
logger.info("=" * 70)

sample_batch = next(iter(train_loader))

print("\nBatch Structure\n")

for key, value in sample_batch.items():

    if isinstance(value, torch.Tensor):

        print(f"{key:<18} {tuple(value.shape)}")

    else:

        print(f"{key:<18} {type(value)}")

print("\nTokenizer Information\n")

print(f"Vocabulary Size : {tokenizer.vocab_size:,}")

print(f"Model           : {MODEL_NAME}")

print(f"Maximum Length  : {MAX_LENGTH}")

print(f"Batch Size      : {BATCH_SIZE}")

logger.info("=" * 70)
logger.info("SECTION 2 COMPLETE")
logger.info("=" * 70)

logger.info(

    "Tokenisation completed successfully."

)

logger.info(

    "Ready for multi-task model construction."

)

In [ ]:
# ==========================================================
# Section 3: Multi-task Model Definition
# ==========================================================
#
# Purpose
# -------
# Define the multi-task XLM-RoBERTa architecture used for
# simultaneous misinformation detection and hate speech
# detection.
#
# Architecture
# ------------
#
#               XLM-RoBERTa Encoder
#                       │
#            CLS / <s> Representation
#                  /             \
#                 /               \
#      Misinformation Head     Hate Speech Head
#          (2 Classes)           (3 Classes)
#
# Outputs
# -------
# multitask_model
#
# ==========================================================

import torch.nn as nn

from transformers import AutoModel

logger.info("=" * 70)
logger.info("MULTI-TASK MODEL DEFINITION")
logger.info("=" * 70)

# ==========================================================
# Multi-task XLM-RoBERTa Model
# ==========================================================

class MultiTaskXLMRoberta(nn.Module):
    """
    Multi-task XLM-RoBERTa classifier.

    Shared encoder

        XLM-RoBERTa Base

    Task-specific heads

        • Misinformation Detection
            2 classes

        • Hate Speech Detection
            3 classes
    """

    def __init__(

        self,

        model_name,

        dropout=0.1

    ):

        super().__init__()

        # --------------------------------------------------
        # Shared Transformer Encoder
        # --------------------------------------------------

        self.encoder = AutoModel.from_pretrained(

            model_name

        )

        hidden_size = (

            self.encoder.config.hidden_size

        )

        # --------------------------------------------------
        # Shared Dropout
        # --------------------------------------------------

        self.dropout = nn.Dropout(

            dropout

        )

        # --------------------------------------------------
        # Task Head:
        # Misinformation Detection
        # --------------------------------------------------

        self.misinformation_classifier = nn.Linear(

            hidden_size,

            2

        )

        # --------------------------------------------------
        # Task Head:
        # Hate Speech Detection
        # --------------------------------------------------

        self.hate_classifier = nn.Linear(

            hidden_size,

            3

        )

    # ======================================================
    # Forward Pass
    # ======================================================

    def forward(

        self,

        input_ids,

        attention_mask

    ):

        outputs = self.encoder(

            input_ids=input_ids,

            attention_mask=attention_mask

        )

        pooled_output = outputs.last_hidden_state[:, 0]

        pooled_output = self.dropout(

            pooled_output

        )

        misinformation_logits = (

            self.misinformation_classifier(

                pooled_output

            )

        )

        hate_logits = (

            self.hate_classifier(

                pooled_output

            )

        )

        return {

            "misinformation":

                misinformation_logits,

            "hate":

                hate_logits

        }

# ==========================================================
# Instantiate Model
# ==========================================================

logger.info("Building multi-task model...")

multitask_model = MultiTaskXLMRoberta(

    model_name=MODEL_NAME,

    dropout=0.1

)

multitask_model.to(DEVICE)

logger.info("✓ Model created successfully.")

# ==========================================================
# Model Summary
# ==========================================================

encoder_parameters = sum(

    p.numel()

    for p in multitask_model.encoder.parameters()

)

misinformation_parameters = sum(

    p.numel()

    for p in multitask_model.misinformation_classifier.parameters()

)

hate_parameters = sum(

    p.numel()

    for p in multitask_model.hate_classifier.parameters()

)

total_parameters = sum(

    p.numel()

    for p in multitask_model.parameters()

)

trainable_parameters = sum(

    p.numel()

    for p in multitask_model.parameters()

    if p.requires_grad

)

print("\nModel Summary\n")

summary = pd.DataFrame({

    "Component": [

        "Shared Encoder",

        "Misinformation Head",

        "Hate Speech Head",

        "Total Parameters",

        "Trainable Parameters"

    ],

    "Parameters": [

        f"{encoder_parameters:,}",

        f"{misinformation_parameters:,}",

        f"{hate_parameters:,}",

        f"{total_parameters:,}",

        f"{trainable_parameters:,}"

    ]

})

print(summary)

# ==========================================================
# Verify Output Dimensions
# ==========================================================

logger.info("=" * 70)
logger.info("MODEL VERIFICATION")
logger.info("=" * 70)

sample_batch = next(iter(train_loader))

multitask_model.eval()

with torch.no_grad():

    outputs = multitask_model(

        input_ids=sample_batch["input_ids"].to(DEVICE),

        attention_mask=sample_batch["attention_mask"].to(DEVICE)

    )

print("\nOutput Shapes\n")

print(

    "Misinformation :",

    tuple(

        outputs["misinformation"].shape

    )

)

print(

    "Hate Speech    :",

    tuple(

        outputs["hate"].shape

    )

)

logger.info("✓ Forward pass successful.")

logger.info("=" * 70)
logger.info("SECTION 3 COMPLETE")
logger.info("=" * 70)

logger.info(

    "Multi-task architecture initialised."

)

logger.info(

    "Ready for loss function configuration."

)

In [ ]:
# ==========================================================
# Section 4: Loss Functions & Optimiser
# ==========================================================
#
# Purpose
# -------
# Configure the optimisation components required for
# multi-task learning.
#
# This section:
#   • Loads task-specific class weights
#   • Configures weighted loss functions
#   • Creates the AdamW optimiser
#   • Configures the learning-rate scheduler
#   • Initialises mixed precision training
#
# Outputs
# -------
# misinformation_loss_fn
# hate_loss_fn
# optimizer
# scheduler
# scaler
#
# ==========================================================

import torch
import torch.nn as nn

from transformers import get_linear_schedule_with_warmup

logger.info("=" * 70)
logger.info("CONFIGURING LOSS FUNCTIONS & OPTIMISER")
logger.info("=" * 70)

# ==========================================================
# Load Task-Specific Class Weights
# ==========================================================

logger.info("Loading class weights...")

misinformation_weights = torch.tensor(

    multitask_class_weights["misinformation"],

    dtype=torch.float

).to(DEVICE)

hate_weights = torch.tensor(

    multitask_class_weights["hate"],

    dtype=torch.float

).to(DEVICE)

logger.info("✓ Class weights loaded.")

print("\nMisinformation Class Weights")

print(misinformation_weights)

print("\nHate Speech Class Weights")

print(hate_weights)

# ==========================================================
# Loss Functions
# ==========================================================

logger.info("Creating weighted loss functions...")

misinformation_loss_fn = nn.CrossEntropyLoss(

    weight=misinformation_weights

)

hate_loss_fn = nn.CrossEntropyLoss(

    weight=hate_weights

)

logger.info("✓ Loss functions created.")

# ==========================================================
# Optimiser
# ==========================================================

logger.info("Creating optimiser...")

optimizer = torch.optim.AdamW(

    multitask_model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)

logger.info("✓ AdamW optimiser initialised.")

# ==========================================================
# Learning Rate Scheduler
# ==========================================================

logger.info("Configuring learning-rate scheduler...")

total_training_steps = len(train_loader) * NUM_EPOCHS

warmup_steps = int(0.10 * total_training_steps)

scheduler = get_linear_schedule_with_warmup(

    optimizer,

    num_warmup_steps=warmup_steps,

    num_training_steps=total_training_steps

)

logger.info("✓ Scheduler configured.")

# ==========================================================
# Mixed Precision Training
# ==========================================================

use_amp = DEVICE.type == "cuda"

if use_amp:

    scaler = torch.cuda.amp.GradScaler()

    logger.info("✓ Mixed precision enabled.")

else:

    scaler = None

    logger.info("Mixed precision disabled (CPU/MPS).")

# ==========================================================
# Training Configuration Summary
# ==========================================================

logger.info("=" * 70)
logger.info("TRAINING CONFIGURATION")
logger.info("=" * 70)

configuration = pd.DataFrame({

    "Parameter": [

        "Model",

        "Device",

        "Batch Size",

        "Learning Rate",

        "Weight Decay",

        "Epochs",

        "Warm-up Steps",

        "Gradient Clip",

        "Optimizer",

        "Scheduler",

        "Mixed Precision",

        "Loss Weight α",

        "Loss Weight β"

    ],

    "Value": [

        MODEL_NAME,

        str(DEVICE),

        BATCH_SIZE,

        LEARNING_RATE,

        WEIGHT_DECAY,

        NUM_EPOCHS,

        warmup_steps,

        GRADIENT_CLIP,

        "AdamW",

        "Linear Warmup",

        use_amp,

        BASELINE_ALPHA,

        BASELINE_BETA

    ]

})

print("\nTraining Configuration\n")

print(configuration)

logger.info("=" * 70)
logger.info("SECTION 4 COMPLETE")
logger.info("=" * 70)

logger.info(

    "Loss functions and optimiser configured successfully."

)

logger.info(

    "Ready for model training."

)

In [ ]:
# ==========================================================
# Section 5: Training Loop
# ==========================================================
#
# Purpose
# -------
# Train the multi-task XLM-RoBERTa model using weighted
# multi-task learning with early stopping and checkpointing.
#
# This section:
#   • Defines training and validation routines
#   • Computes task-specific losses
#   • Combines losses using α and β
#   • Tracks learning curves
#   • Performs early stopping
#   • Saves the best-performing model
#
# Outputs
# -------
# best_model.pt
# training_history.csv
#
# ==========================================================

from copy import deepcopy
from tqdm.auto import tqdm

logger.info("=" * 70)
logger.info("MODEL TRAINING")
logger.info("=" * 70)

# ==========================================================
# Training Epoch
# ==========================================================

def train_epoch(model, dataloader):

    model.train()

    running_loss = 0.0

    running_misinfo_loss = 0.0

    running_hate_loss = 0.0

    progress_bar = tqdm(

        dataloader,

        desc="Training",

        leave=False

    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(DEVICE)

        attention_mask = batch["attention_mask"].to(DEVICE)

        task_ids = batch["task_id"].to(DEVICE)

        task_labels = batch["task_label"].to(DEVICE)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask

            )

            total_loss = torch.tensor(

                0.0,

                device=DEVICE

            )

            misinfo_loss = torch.tensor(

                0.0,

                device=DEVICE

            )

            hate_loss = torch.tensor(

                0.0,

                device=DEVICE

            )

            # ---------------------------------------------
            # Misinformation Task
            # ---------------------------------------------

            misinfo_mask = task_ids == 0

            if misinfo_mask.any():

                misinfo_loss = misinformation_loss_fn(

                    outputs["misinformation"][misinfo_mask],

                    task_labels[misinfo_mask]

                )

                total_loss += (

                    BASELINE_ALPHA *

                    misinfo_loss

                )

            # ---------------------------------------------
            # Hate Speech Task
            # ---------------------------------------------

            hate_mask = task_ids == 1

            if hate_mask.any():

                hate_loss = hate_loss_fn(

                    outputs["hate"][hate_mask],

                    task_labels[hate_mask]

                )

                total_loss += (

                    BASELINE_BETA *

                    hate_loss

                )

        if use_amp:

            scaler.scale(total_loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                GRADIENT_CLIP

            )

            scaler.step(optimizer)

            scaler.update()

        else:

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                GRADIENT_CLIP

            )

            optimizer.step()

        scheduler.step()

        running_loss += total_loss.item()

        running_misinfo_loss += misinfo_loss.item()

        running_hate_loss += hate_loss.item()

        progress_bar.set_postfix(

            loss=f"{total_loss.item():.4f}"

        )

    return {

        "loss":

            running_loss / len(dataloader),

        "misinfo_loss":

            running_misinfo_loss / len(dataloader),

        "hate_loss":

            running_hate_loss / len(dataloader)

    }

# ==========================================================
# Validation Epoch
# ==========================================================

def validate_epoch(model, dataloader):

    model.eval()

    running_loss = 0.0

    running_misinfo_loss = 0.0

    running_hate_loss = 0.0

    with torch.no_grad():

        for batch in tqdm(

            dataloader,

            desc="Validation",

            leave=False

        ):

            input_ids = batch["input_ids"].to(DEVICE)

            attention_mask = batch["attention_mask"].to(DEVICE)

            task_ids = batch["task_id"].to(DEVICE)

            task_labels = batch["task_label"].to(DEVICE)

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask

            )

            total_loss = torch.tensor(

                0.0,

                device=DEVICE

            )

            misinfo_loss = torch.tensor(

                0.0,

                device=DEVICE

            )

            hate_loss = torch.tensor(

                0.0,

                device=DEVICE

            )

            misinfo_mask = task_ids == 0

            if misinfo_mask.any():

                misinfo_loss = misinformation_loss_fn(

                    outputs["misinformation"][misinfo_mask],

                    task_labels[misinfo_mask]

                )

                total_loss += (

                    BASELINE_ALPHA *

                    misinfo_loss

                )

            hate_mask = task_ids == 1

            if hate_mask.any():

                hate_loss = hate_loss_fn(

                    outputs["hate"][hate_mask],

                    task_labels[hate_mask]

                )

                total_loss += (

                    BASELINE_BETA *

                    hate_loss

                )

            running_loss += total_loss.item()

            running_misinfo_loss += misinfo_loss.item()

            running_hate_loss += hate_loss.item()

    return {

        "loss":

            running_loss / len(dataloader),

        "misinfo_loss":

            running_misinfo_loss / len(dataloader),

        "hate_loss":

            running_hate_loss / len(dataloader)

    }

# ==========================================================
# Training Loop
# ==========================================================

history = []

best_validation_loss = float("inf")

best_epoch = 0

patience_counter = 0

best_state_dict = None

logger.info("Starting training...\n")

for epoch in range(NUM_EPOCHS):

    logger.info(

        f"Epoch {epoch + 1}/{NUM_EPOCHS}"

    )

    train_metrics = train_epoch(

        multitask_model,

        train_loader

    )

    validation_metrics = validate_epoch(

        multitask_model,

        validation_loader

    )

    history.append({

        "epoch":

            epoch + 1,

        "train_loss":

            train_metrics["loss"],

        "validation_loss":

            validation_metrics["loss"],

        "train_misinfo_loss":

            train_metrics["misinfo_loss"],

        "validation_misinfo_loss":

            validation_metrics["misinfo_loss"],

        "train_hate_loss":

            train_metrics["hate_loss"],

        "validation_hate_loss":

            validation_metrics["hate_loss"]

    })

    logger.info(

        f"Train Loss: "

        f"{train_metrics['loss']:.4f}"

    )

    logger.info(

        f"Validation Loss: "

        f"{validation_metrics['loss']:.4f}"

    )

    # -----------------------------------------
    # Save Best Model
    # -----------------------------------------

    if validation_metrics["loss"] < best_validation_loss:

        best_validation_loss = validation_metrics["loss"]

        best_epoch = epoch + 1

        patience_counter = 0

        best_state_dict = deepcopy(

            multitask_model.state_dict()

        )

        torch.save(

            best_state_dict,

            CHECKPOINT_DIR /

            "best_model.pt"

        )

        logger.info(

            "✓ Best model updated."

        )

    else:

        patience_counter += 1

        logger.info(

            f"Early stopping patience: "

            f"{patience_counter}/"

            f"{EARLY_STOPPING_PATIENCE}"

        )

        if (

            patience_counter >=

            EARLY_STOPPING_PATIENCE

        ):

            logger.info(

                "Early stopping triggered."

            )

            break

# ==========================================================
# Restore Best Model
# ==========================================================

multitask_model.load_state_dict(

    best_state_dict

)

training_history = pd.DataFrame(history)

training_history.to_csv(

    BASELINE_DIR /

    "training_history.csv",

    index=False

)

logger.info("=" * 70)

logger.info(

    f"Best Epoch : {best_epoch}"

)

logger.info(

    f"Best Validation Loss : "

    f"{best_validation_loss:.4f}"

)

logger.info("=" * 70)

logger.info("SECTION 5 COMPLETE")

logger.info("=" * 70)

logger.info(

    "Training completed successfully."

)

logger.info(

    "Best model checkpoint saved."

)

In [ ]:
# ==========================================================
# Section 6: Final Evaluation
# ==========================================================
#
# Purpose
# -------
# Evaluate the best-performing multi-task model on the
# held-out test dataset.
#
# This section:
#   • Generates predictions
#   • Computes evaluation metrics
#   • Produces confusion matrices
#   • Stores evaluation artefacts
#
# Outputs
# -------
# evaluation_results
# confusion_matrices
#
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

logger.info("=" * 70)
logger.info("FINAL MODEL EVALUATION")
logger.info("=" * 70)

multitask_model.eval()

misinfo_true = []
misinfo_pred = []

hate_true = []
hate_pred = []

with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Testing"
    ):

        input_ids = batch["input_ids"].to(DEVICE)

        attention_mask = batch["attention_mask"].to(DEVICE)

        task_ids = batch["task_id"].to(DEVICE)

        task_labels = batch["task_label"].to(DEVICE)

        outputs = multitask_model(

            input_ids=input_ids,

            attention_mask=attention_mask

        )

        # ----------------------------------------------
        # Misinformation Predictions
        # ----------------------------------------------

        misinfo_mask = task_ids == 0

        if misinfo_mask.any():

            predictions = torch.argmax(

                outputs["misinformation"][misinfo_mask],

                dim=1

            )

            misinfo_true.extend(

                task_labels[misinfo_mask]

                .cpu()

                .numpy()

            )

            misinfo_pred.extend(

                predictions

                .cpu()

                .numpy()

            )

        # ----------------------------------------------
        # Hate Speech Predictions
        # ----------------------------------------------

        hate_mask = task_ids == 1

        if hate_mask.any():

            predictions = torch.argmax(

                outputs["hate"][hate_mask],

                dim=1

            )

            hate_true.extend(

                task_labels[hate_mask]

                .cpu()

                .numpy()

            )

            hate_pred.extend(

                predictions

                .cpu()

                .numpy()

            )

logger.info("Predictions generated successfully.")

# ==========================================================
# Evaluation Function
# ==========================================================

def evaluate_task(

    y_true,

    y_pred,

    task_name,

    target_names

):

    accuracy = accuracy_score(

        y_true,

        y_pred

    )

    precision,

    recall,

    f1,

    _ = precision_recall_fscore_support(

        y_true,

        y_pred,

        average="macro",

        zero_division=0

    )

    report = classification_report(

        y_true,

        y_pred,

        target_names=target_names,

        output_dict=True,

        zero_division=0

    )

    matrix = confusion_matrix(

        y_true,

        y_pred

    )

    logger.info("=" * 70)

    logger.info(task_name)

    logger.info("=" * 70)

    print(

        classification_report(

            y_true,

            y_pred,

            target_names=target_names,

            zero_division=0

        )

    )

    return {

        "accuracy": float(accuracy),

        "precision_macro": float(precision),

        "recall_macro": float(recall),

        "f1_macro": float(f1),

        "classification_report": report,

        "confusion_matrix": matrix.tolist()

    }

# ==========================================================
# Evaluate Misinformation Task
# ==========================================================

misinformation_results = evaluate_task(

    misinfo_true,

    misinfo_pred,

    "MISINFORMATION DETECTION",

    [

        "Not Misinformation",

        "Misinformation"

    ]

)

# ==========================================================
# Evaluate Hate Speech Task
# ==========================================================

hate_results = evaluate_task(

    hate_true,

    hate_pred,

    "HATE SPEECH DETECTION",

    [

        "Normal",

        "Abusive",

        "Hate"

    ]

)

# ==========================================================
# Overall Evaluation Summary
# ==========================================================

evaluation_results = {

    "misinformation":

        misinformation_results,

    "hate":

        hate_results

}

summary = pd.DataFrame({

    "Task": [

        "Misinformation",

        "Hate Speech"

    ],

    "Accuracy": [

        misinformation_results["accuracy"],

        hate_results["accuracy"]

    ],

    "Macro Precision": [

        misinformation_results["precision_macro"],

        hate_results["precision_macro"]

    ],

    "Macro Recall": [

        misinformation_results["recall_macro"],

        hate_results["recall_macro"]

    ],

    "Macro F1": [

        misinformation_results["f1_macro"],

        hate_results["f1_macro"]

    ]

})

logger.info("=" * 70)
logger.info("FINAL TEST RESULTS")
logger.info("=" * 70)

print(summary)

logger.info("=" * 70)
logger.info("SECTION 6 COMPLETE")
logger.info("=" * 70)

logger.info("Final evaluation completed successfully.")

In [ ]:
# ==========================================================
# Section 7: Export Training Artefacts
# ==========================================================
#
# Purpose
# -------
# Export all artefacts required to reproduce the baseline
# multi-task experiment.
#
# This section:
#   • Saves the trained tokenizer
#   • Saves evaluation metrics
#   • Saves training configuration
#   • Saves confusion matrices
#   • Performs final validation
#
# Outputs
# -------
# models/
#   baseline/
#       checkpoints/
#           best_model.pt
#
#       tokenizer/
#
#       figures/
#
#       training_history.csv
#       evaluation_metrics.json
#       training_config.json
#       confusion_matrices.json
#
# ==========================================================

import json

logger.info("=" * 70)
logger.info("EXPORTING TRAINING ARTEFACTS")
logger.info("=" * 70)

# ==========================================================
# Save Tokenizer
# ==========================================================

logger.info("Saving tokenizer...")

TOKENIZER_DIR = BASELINE_DIR / "tokenizer"

TOKENIZER_DIR.mkdir(

    parents=True,

    exist_ok=True

)

tokenizer.save_pretrained(

    TOKENIZER_DIR

)

logger.info("✓ Tokenizer saved.")

# ==========================================================
# Save Evaluation Metrics
# ==========================================================

logger.info("Saving evaluation metrics...")

evaluation_path = (

    BASELINE_DIR /

    "evaluation_metrics.json"

)

with open(

    evaluation_path,

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        evaluation_results,

        fp,

        indent=4

    )

logger.info("✓ Evaluation metrics saved.")

# ==========================================================
# Save Training Configuration
# ==========================================================

logger.info("Saving training configuration...")

config_path = (

    BASELINE_DIR /

    "training_config.json"

)

with open(

    config_path,

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        training_config,

        fp,

        indent=4

    )

logger.info("✓ Training configuration saved.")

# ==========================================================
# Save Confusion Matrices
# ==========================================================

logger.info("Saving confusion matrices...")

confusion_path = (

    BASELINE_DIR /

    "confusion_matrices.json"

)

confusion_data = {

    "misinformation":

        misinformation_results[

            "confusion_matrix"

        ],

    "hate":

        hate_results[

            "confusion_matrix"

        ]

}

with open(

    confusion_path,

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        confusion_data,

        fp,

        indent=4

    )

logger.info("✓ Confusion matrices saved.")

# ==========================================================
# Final Validation
# ==========================================================

logger.info("=" * 70)
logger.info("FINAL VALIDATION")
logger.info("=" * 70)

expected_outputs = [

    CHECKPOINT_DIR /

    "best_model.pt",

    BASELINE_DIR /

    "training_history.csv",

    BASELINE_DIR /

    "evaluation_metrics.json",

    BASELINE_DIR /

    "training_config.json",

    BASELINE_DIR /

    "confusion_matrices.json",

    TOKENIZER_DIR

]

missing_outputs = [

    path

    for path in expected_outputs

    if not path.exists()

]

if missing_outputs:

    logger.warning(

        "Some expected outputs were not found:"

    )

    for file in missing_outputs:

        logger.warning(f"  ✗ {file}")

else:

    logger.info(

        "✓ All expected artefacts successfully generated."

    )

# ==========================================================
# Completion Summary
# ==========================================================

logger.info("=" * 70)
logger.info("BASELINE TRAINING COMPLETE")
logger.info("=" * 70)

summary = pd.DataFrame({

    "Artefact": [

        "Best Model",

        "Tokenizer",

        "Training History",

        "Evaluation Metrics",

        "Training Configuration",

        "Confusion Matrices"

    ],

    "Status": [

        "✓ Saved",

        "✓ Saved",

        "✓ Saved",

        "✓ Saved",

        "✓ Saved",

        "✓ Saved"

    ]

})

print("\nGenerated Artefacts\n")

print(summary)

print("\nSaved to\n")

print(BASELINE_DIR)

logger.info("Notebook completed successfully.")

logger.info(

    "Baseline model ready for comparison with the "

    "misinformation-tuned experiment."

)